
# Day 14 — Level 2 Modeling Preparation

## Objective

Prepare a reproducible modeling workflow for evaluating Level 2 anonymized data against baseline and Level 1 results. This notebook documents the experiment design and validates the dataset schema; model retraining will be performed in the subsequent modeling tasks.

## Dataset Path and Schema Validation

### Validation Findings

- Level 1 contains 32,561 rows and Level 2 contains 32,560 rows.
- Both datasets contain 16 columns.
- Level 2 uses generalized age bands and masked ZIP codes.
- The income target contains no missing values.
- Missing values remain in workclass, occupation, and native_country.
- Missing feature values must be handled by preprocessing fitted on the training data only.
- The reason for the one-row difference must be verified before attributing it to suppression.

The original datasets remain unchanged during validation.


### Target Distribution Comparison

| Metric | Level 1 | Level 2 |
|---|---:|---:|
| Total rows | 32,561 | 32,560 |
| Income <=50K | 24,720 | 24,719 |
| Income >50K | 7,841 | 7,841 |
| Proportion <=50K | 75.92% | 75.92% |
| Proportion >50K | 24.08% | 24.08% |

The displayed income-class proportions are unchanged after rounding to four decimal places. Level 2 contains one fewer record in the <=50K class, while the >50K count is unchanged. The reason for the row difference remains to be verified.


## Target and Feature Definition

### Target and Feature Definition

- **Target column:** `income`
- **Classification task:** Binary classification of income category.
- **Target classes:** `<=50K` and `>50K`
- **Feature columns:** All other columns, excluding `income`.
- **Excluded identifier:** None at this stage; the dataset schema will be reviewed for any columns that should not be used as predictive features.

The same target definition and feature-selection logic will be used for the baseline, Level 1, and Level 2 experiments to maintain comparability.

## Preprocessing Plan

### Preprocessing Strategy

1. Separate the target column (`income`) from the feature columns.
2. Use a consistent train/test split strategy across baseline, Level 1, and Level 2 experiments.
3. Identify numeric and categorical feature columns.
4. Impute missing numeric values using the median, fitted only on the training data.
5. Impute missing categorical values using the most frequent category, fitted only on the training data.
6. Encode categorical features using a consistent encoding strategy.
7. Scale numeric features where appropriate for Logistic Regression.
8. Fit all preprocessing steps on training data only, then transform the test data using the fitted pipeline.
9. Keep the test set separate from all fitting and model-selection steps.

### Comparability Requirements

- Use the same target definition and evaluation metrics across experiments.
- Reuse the established train/test split and random seed where available.
- Do not modify the original datasets during preprocessing.
- Document any differences in row count or feature representation.

## Logistic Regression and Random Forest Experiment Plan

### Model Configuration

Two classification models will be evaluated on the Level 2 anonymized dataset.

#### 1. Logistic Regression

- Model: `LogisticRegression`
- `max_iter=1000`
- `random_state=42`
- Numeric features will be imputed and scaled.
- Categorical features will be imputed and one-hot encoded.

#### 2. Random Forest

- Model: `RandomForestClassifier`
- `n_estimators=100`
- `random_state=42`
- `n_jobs=-1`
- The same training and test split will be used for both models.

### Experimental Procedure

1. Load the Level 2 dataset without modifying the original file.
2. Separate `income` from the feature columns.
3. Create an 80/20 stratified train/test split using `random_state=42`.
4. Fit preprocessing steps on the training data only.
5. Train Logistic Regression and Random Forest.
6. Generate predictions on the held-out test set.
7. Record the evaluation metrics and compare them with the baseline and Level 1 results.

## Evaluation Metrics

### Evaluation Metrics

Both models will be evaluated using the following metrics:

- **Accuracy:** Overall proportion of correct predictions.
- **Precision:** Proportion of predicted `>50K` cases that are actually `>50K`.
- **Recall:** Proportion of actual `>50K` cases correctly identified.
- **F1-score:** Harmonic mean of precision and recall.

The positive class will be `>50K`.

Confusion matrices will also be generated to examine classification errors.

Results will be recorded in a structured format and compared against the existing baseline and Level 1 metrics. Any differences in dataset size, preprocessing, or experimental settings will be documented.

## Baseline and Level 1 Comparison

### Existing Model Results

The following results were recorded in the baseline and Level 1 experiments.

#### Logistic Regression

| Metric | Baseline | Level 1 |
|---|---:|---:|
| Accuracy | 0.855059 | 0.855520 |
| Precision | 0.740370 | 0.730345 |
| Recall | 0.612883 | 0.633929 |
| F1-score | 0.670621 | 0.678730 |

#### Random Forest

| Metric | Baseline | Level 1 |
|---|---:|---:|
| Accuracy | 0.860740 | 0.861508 |
| Precision | 0.770680 | 0.760972 |
| Recall | 0.600128 | 0.619260 |
| F1-score | 0.674794 | 0.682841 |

### Preliminary Interpretation

- Both models show a small increase in accuracy and F1-score in Level 1 compared with the baseline.
- Recall increases for both models, while precision decreases.
- These results describe model utility; they do not independently establish the privacy protection achieved by anonymization.
- Level 2 results will be compared with both baseline and Level 1 using the same evaluation metrics.

The comparison assumes the experiments use consistent train/test splits, preprocessing, and evaluation procedures. These settings should be verified against the existing experiment code before drawing final conclusions.

## Pending Checks and Team Dependencies

### Pending Checks

- [ ] Verify the existing baseline and Level 1 train/test split and random seed.
- [ ] Confirm that preprocessing and model settings match the existing experiments.
- [ ] Investigate why Level 2 contains one fewer row than Level 1.
- [ ] Confirm how missing values in workclass, occupation, and native_country are handled.
- [ ] Verify the provenance and intended use of the ZIP-code feature.
- [ ] Confirm that Level 2 anonymization settings are documented.
- [ ] Train and evaluate Logistic Regression and Random Forest on Level 2.
- [ ] Compare Level 2 results with baseline and Level 1.
- [ ] Record results and limitations for the team handover.

### Team Dependencies

The Level 2 dataset is available at:

`data/anonymized/level2/adult_level2_medium.csv`

The dataset has been loaded and its schema inspected. The modeling experiment can proceed once the existing experimental settings and the one-row difference have been checked.

### Verified Level 2 Experiment Implementation

The existing `src/level2_experiment.py` script trains and evaluates Logistic Regression and Random Forest for the configured low, medium, and high anonymization levels. It uses an 80/20 stratified train-test split with `random_state=42`, median imputation and scaling for numeric features, most-frequent imputation and one-hot encoding for categorical features, and the baseline model settings.

The saved privacy results confirm that the medium level suppresses one record to achieve a minimum k-anonymity of 3. The high level achieves a minimum k-anonymity of 6 without suppression. The medium-level age generalization width is 10 years, matching the low-level setting; the high level uses 20-year age bands.

Before final reporting, verify the experiment outputs and document that the baseline, Level 1, and Level 2 comparisons use consistent evaluation settings. Avoid attributing model performance changes to anonymization alone without acknowledging the one-record difference and any differences in the anonymized feature representations.

### Expected Deliverables

- Validated Level 2 dataset documentation.
- Reproducible modeling pipeline.
- Logistic Regression and Random Forest evaluation metrics.
- Baseline, Level 1, and Level 2 comparison.
- Documented privacy and utility limitations.


In [1]:

from pathlib import Path
import pandas as pd

# Find available anonymized datasets
data_dir = Path("../data/anonymized")

for path in sorted(data_dir.rglob("*.csv")):
    print(f"{path} | {path.stat().st_size:,} bytes")


..\data\anonymized\level1\adult_level1.csv | 3,839,964 bytes
..\data\anonymized\level2\adult_level2_medium.csv | 3,839,849 bytes


In [2]:
# dataset validation code cell


from pathlib import Path
import pandas as pd

# Dataset paths relative to the notebooks folder
level1_path = Path("../data/anonymized/level1/adult_level1.csv")
level2_path = Path("../data/anonymized/level2/adult_level2_medium.csv")

# Load both datasets
level1_df = pd.read_csv(level1_path)
level2_df = pd.read_csv(level2_path)

# Inspect dataset dimensions
print("LEVEL 1 DATASET")
print("Shape:", level1_df.shape)

print("\nLEVEL 2 DATASET")
print("Shape:", level2_df.shape)

# Inspect columns and data types
print("\nLEVEL 2 COLUMNS AND DATA TYPES")
print(level2_df.dtypes)

# Validate target column
print("\nTARGET VALIDATION")
print("'income' exists:", "income" in level2_df.columns)

if "income" in level2_df.columns:
    print("Missing target values:", level2_df["income"].isna().sum())
    print("\nTarget distribution:")
    print(level2_df["income"].value_counts(dropna=False))

# Check whether schemas match
print("\nSCHEMA COMPARISON")
print("Same column names:", set(level1_df.columns) == set(level2_df.columns))
print("Level 1 only:", sorted(set(level1_df.columns) - set(level2_df.columns)))
print("Level 2 only:", sorted(set(level2_df.columns) - set(level1_df.columns)))

# Check missing values in Level 2
print("\nLEVEL 2 MISSING VALUES")
print(level2_df.isna().sum().sort_values(ascending=False))


LEVEL 1 DATASET
Shape: (32561, 16)

LEVEL 2 DATASET
Shape: (32560, 16)

LEVEL 2 COLUMNS AND DATA TYPES
age                 str
workclass           str
fnlwgt            int64
education           str
education_num     int64
marital_status      str
occupation          str
relationship        str
race                str
sex                 str
capital_gain      int64
capital_loss      int64
hours_per_week    int64
native_country      str
income              str
zip_code            str
dtype: object

TARGET VALIDATION
'income' exists: True
Missing target values: 0

Target distribution:
income
<=50K    24719
>50K      7841
Name: count, dtype: int64

SCHEMA COMPARISON
Same column names: True
Level 1 only: []
Level 2 only: []

LEVEL 2 MISSING VALUES
occupation        1843
workclass         1836
native_country     583
fnlwgt               0
education            0
education_num        0
marital_status       0
age                  0
relationship         0
race                 0
capital_gain     

In [3]:

# Compare selected columns between Level 1 and Level 2

print("LEVEL 1 SAMPLE")
display(level1_df[["age", "zip_code", "income"]].head(10))

print("\nLEVEL 2 SAMPLE")
display(level2_df[["age", "zip_code", "income"]].head(10))

print("\nAGE VALUES — LEVEL 2")
print(level2_df["age"].value_counts(dropna=False).head(20))

print("\nZIP CODE VALUES — LEVEL 2")
print(level2_df["zip_code"].value_counts(dropna=False).head(20))

print("\nROW DUPLICATES")
print("Level 1 duplicate rows:", level1_df.duplicated().sum())
print("Level 2 duplicate rows:", level2_df.duplicated().sum())

print("\nMISSING VALUES — ALL LEVEL 2 COLUMNS")
print(level2_df.isna().sum().to_string())


LEVEL 1 SAMPLE


,age,zip_code,income
0,30-39,938**,<=50K
1,50-59,245**,<=50K
2,30-39,132**,<=50K
3,50-59,460**,<=50K
4,20-29,420**,<=50K
5,30-39,392**,<=50K
6,40-49,282**,<=50K
7,50-59,234**,>50K
8,30-39,986**,>50K
9,40-49,814**,>50K



LEVEL 2 SAMPLE


,age,zip_code,income
0,30-39,9****,<=50K
1,50-59,2****,<=50K
2,30-39,1****,<=50K
3,50-59,4****,<=50K
4,20-29,4****,<=50K
5,30-39,3****,<=50K
6,40-49,2****,<=50K
7,50-59,2****,>50K
8,30-39,9****,>50K
9,40-49,8****,>50K



AGE VALUES — LEVEL 2
age
30-39    8613
20-29    8054
40-49    7175
50-59    4418
60-69    2015
10-19    1657
70-79     508
80-89      78
90-99      42
Name: count, dtype: int64

ZIP CODE VALUES — LEVEL 2
zip_code
2****    3786
8****    3718
6****    3706
3****    3630
1****    3586
9****    3576
4****    3556
5****    3520
7****    3482
Name: count, dtype: int64

ROW DUPLICATES
Level 1 duplicate rows: 0
Level 2 duplicate rows: 10

MISSING VALUES — ALL LEVEL 2 COLUMNS
age                  0
workclass         1836
fnlwgt               0
education            0
education_num        0
marital_status       0
occupation        1843
relationship         0
race                 0
sex                  0
capital_gain         0
capital_loss         0
hours_per_week       0
native_country     583
income               0
zip_code             0


In [4]:

# 1. Compare dataset sizes
print("Level 1 rows:", len(level1_df))
print("Level 2 rows:", len(level2_df))
print("Rows removed:", len(level1_df) - len(level2_df))

# 2. Check missing values across every Level 2 column
print("\nMissing values in Level 2:")
print(level2_df.isna().sum().to_string())

# 3. Compare age and ZIP representations
print("\nLevel 1 age examples:")
print(level1_df["age"].dropna().unique()[:10])

print("\nLevel 2 age bands:")
print(sorted(level2_df["age"].dropna().unique()))

print("\nLevel 1 ZIP examples:")
print(level1_df["zip_code"].dropna().astype(str).unique()[:10])

print("\nLevel 2 ZIP examples:")
print(level2_df["zip_code"].dropna().astype(str).unique()[:10])

# 4. Check whether the target distribution changed
print("\nLevel 1 income distribution:")
print(level1_df["income"].value_counts(normalize=True).round(4))

print("\nLevel 2 income distribution:")
print(level2_df["income"].value_counts(normalize=True).round(4))

# 5. Check whether all Level 2 age bands and ZIP values are strings
print("\nAge values are strings:", level2_df["age"].dropna().map(
    lambda x: isinstance(x, str)
).all())

print("ZIP values are strings:", level2_df["zip_code"].dropna().map(
    lambda x: isinstance(x, str)
).all())


Level 1 rows: 32561
Level 2 rows: 32560
Rows removed: 1

Missing values in Level 2:
age                  0
workclass         1836
fnlwgt               0
education            0
education_num        0
marital_status       0
occupation        1843
relationship         0
race                 0
sex                  0
capital_gain         0
capital_loss         0
hours_per_week       0
native_country     583
income               0
zip_code             0

Level 1 age examples:
<ArrowStringArray>
['30-39', '50-59', '20-29', '40-49', '10-19', '70-79', '60-69', '90-99',
 '80-89']
Length: 9, dtype: str

Level 2 age bands:
['10-19', '20-29', '30-39', '40-49', '50-59', '60-69', '70-79', '80-89', '90-99']

Level 1 ZIP examples:
<ArrowStringArray>
['938**', '245**', '132**', '460**', '420**', '392**', '282**', '234**',
 '986**', '814**']
Length: 10, dtype: str

Level 2 ZIP examples:
<ArrowStringArray>
['9****', '2****', '1****', '4****', '3****', '8****', '6****', '7****',
 '5****']
Length: 9, dtype:

In [5]:

print("LEVEL 1 INCOME COUNTS")
print(level1_df["income"].value_counts(dropna=False))

print("\nLEVEL 2 INCOME COUNTS")
print(level2_df["income"].value_counts(dropna=False))

print("\nLEVEL 1 INCOME PROPORTIONS")
print(level1_df["income"].value_counts(normalize=True).round(4))

print("\nLEVEL 2 INCOME PROPORTIONS")
print(level2_df["income"].value_counts(normalize=True).round(4))

print("\nLEVEL 2 ZIP EXAMPLES")
print(level2_df["zip_code"].value_counts().head(10))

print("\nLEVEL 2 UNIQUE AGE BANDS")
print(sorted(level2_df["age"].unique()))


LEVEL 1 INCOME COUNTS
income
<=50K    24720
>50K      7841
Name: count, dtype: int64

LEVEL 2 INCOME COUNTS
income
<=50K    24719
>50K      7841
Name: count, dtype: int64

LEVEL 1 INCOME PROPORTIONS
income
<=50K    0.7592
>50K     0.2408
Name: proportion, dtype: float64

LEVEL 2 INCOME PROPORTIONS
income
<=50K    0.7592
>50K     0.2408
Name: proportion, dtype: float64

LEVEL 2 ZIP EXAMPLES
zip_code
2****    3786
8****    3718
6****    3706
3****    3630
1****    3586
9****    3576
4****    3556
5****    3520
7****    3482
Name: count, dtype: int64

LEVEL 2 UNIQUE AGE BANDS
['10-19', '20-29', '30-39', '40-49', '50-59', '60-69', '70-79', '80-89', '90-99']
